# Knowledge Distillation: ConvNeXt V2 → MobileNetV3 (Colab)

**Mục tiêu:** Dùng ID embedding 512-D của teacher ConvNeXt V2 (đã train MTL) để hướng dẫn student MobileNetV3 học compact embedding cho face recognition.

| | Teacher | Student |
|---|---|---|
| Model | `MTLFaceRecognition` (ConvNeXt V2) | `FaceRecognitionMobileNetV3` |
| Params | ~28M | ~5M |
| Embedding dùng cho KD | `x_id` (512-D, tầng cuối) | 512-D |
| Mode | **Frozen** | **Trainable** |

**KD Loss:**
```
L_total = α · L_MagFace(student)  +  β · L_KD
L_KD    = mean(1 - cosine_sim(norm(s_emb), norm(t_emb)))
```

**Cách dùng:**
1. Mount Google Drive (cell 1)
2. Sửa `CONFIGURATION` và `TEACHER_CKPT` (cell Config)
3. Chạy từ trên xuống

## 1. Mount Drive & Setup môi trường

In [1]:
!nvidia-smi

Sun May 24 08:56:51 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   34C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
print("Hello, World!")

Hello, World!


In [ ]:
from google.colab import drive
import os

# Mount Google Drive
drive.mount('/content/drive')

REPO_URL    = 'https://github.com/NguyenXuanBinh22/DATN.git'
REPO_BRANCH = 'convnext-v2-dev'
REPO_DIR    = '/content/FR_Photometric_Stereo'

if not os.path.exists(REPO_DIR):
    os.system(f'git clone -b {REPO_BRANCH} {REPO_URL} {REPO_DIR}')
else:
    os.system(f'git -C {REPO_DIR} pull origin {REPO_BRANCH}')
    print('Repo đã tồn tại, đã pull latest.')

%cd {REPO_DIR}
print(f'Working dir: {os.getcwd()}')

os.system('pip install -q albumentations==1.3.1 timm tabulate termcolor')

## 2. Imports & Cấu hình

In [ ]:
%cd /content/FR_Photometric_Stereo
import warnings
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd
import albumentations as A
from torch.optim import Adam
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts
from torch.utils.tensorboard import SummaryWriter
from tabulate import tabulate

from going_modular.dataloader.multitask import create_multitask_datafetcher, create_eval_loaders
from going_modular.model.MTLFaceRecognition import MTLFaceRecognition
from going_modular.model.FaceRecognitionMobileNetV3 import FaceRecognitionMobileNetV3
from going_modular.loss.WeightClassMagLoss import WeightClassMagLoss
from going_modular.utils.transforms import RandomResizedCropRect, GaussianNoise
from going_modular.utils.roc_auc_id import (
    compute_id_auc, compute_rank1,
    compute_id_auc_gallery_probe, compute_rank1_gallery_probe,
)
from going_modular.utils.MultiMetricEarlyStopping import MultiMetricEarlyStopping
from going_modular.utils.ModelCheckPoint import ModelCheckpoint
from going_modular.utils.ExperimentManager import ExperimentManager

device = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.manual_seed(42)
print(f'Device: {device}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [5]:
# ════════════════════════════════════════════════════════════
#  CẤU HÌNH — chỉnh sửa ở đây
# ════════════════════════════════════════════════════════════

# Đường dẫn dataset trên Google Drive
DRIVE_DATASET_DIR = '/content/drive/MyDrive/Photometric_DB_Full/'

# Checkpoint teacher ConvNeXt V2 đã train (best_model.pth hoặc last_model.pth)
# Ví dụ: '/content/drive/MyDrive/experiments/Single_ALBEDO_PK_SAMPLER_ConvNextV2/checkpoints/best_model.pth'
TEACHER_CKPT = '/content/drive/MyDrive/Photometric_DB_Full/experiments/Single_ALBEDO_PK_SAMPLER_ConvNextV2/checkpoints/best_model.pth'

EXPERIMENT_NAME = 'KD_ConvNextV2_to_MobileNetV3_Albedo'

CONFIGURATION = {
    'note':        EXPERIMENT_NAME,
    'dataset_dir': DRIVE_DATASET_DIR,

    # Ghi checkpoint và log vào Drive để không mất khi Colab disconnect
    'output_dir':  '/content/drive/MyDrive/',

    # Modality: 'albedo' | 'normalmap' | 'depthmap'
    # Phải giống modality mà teacher đã được train
    'type':        'albedo',

    # Teacher backbone (để tái tạo đúng kiến trúc khi load checkpoint)
    'teacher_backbone': 'convnextv2_tiny',

    # Student backbone
    'backbone':    'mobilenetv3_large_100',

    'use_sampler': True,   # PK Sampler — mỗi batch gồm batch_size identity khác nhau
    'device':      device,
    'epochs':      40,
    'batch_size':  32,
    'image_size':  112,
    'base_lr':     1e-4,
    'num_classes': None,   # tự động lấy từ CSV

    # Trọng số loss
    # L_total = task_weight * L_MagFace + kd_weight * L_KD
    'task_weight': 1.0,
    'kd_weight':   1.0,
}

print(f"Dataset dir : {CONFIGURATION['dataset_dir']}")
print(f"Output dir  : {CONFIGURATION['output_dir']}")
print(f"Teacher ckpt: {TEACHER_CKPT}")

Dataset dir : /content/drive/MyDrive/Photometric_DB_Full/
Output dir  : /content/drive/MyDrive/
Teacher ckpt: /content/drive/MyDrive/Photometric_DB_Full/experiments/Single_ALBEDO_PK_SAMPLER_ConvNextV2/checkpoints/best_model.pth


## 3. Data Loading

Dùng **cùng transform** với ConvNeXt V2 để student và teacher nhận cùng input trong mỗi batch.

In [ ]:
dataset_dir = CONFIGURATION['dataset_dir']

# Tìm CSV — hỗ trợ cả hai cấu trúc thư mục
train_csv = os.path.join(dataset_dir, 'train_split.csv')
if not os.path.exists(train_csv):
    train_csv = os.path.join(dataset_dir, 'dataset', 'train_split.csv')
if not os.path.exists(train_csv):
    train_csv = os.path.join(dataset_dir, 'train_set.csv')
if not os.path.exists(train_csv):
    raise FileNotFoundError(
        f'Không tìm thấy CSV train tại {dataset_dir}.\n'
        f'Kiểm tra lại DRIVE_DATASET_DIR.'
    )
print(f'Train CSV: {train_csv}')

df_train = pd.read_csv(train_csv)
CONFIGURATION['num_classes'] = int(df_train['id'].nunique())
print(f'num_classes : {CONFIGURATION["num_classes"]}')
print(f'Số mẫu train: {len(df_train)}')

# Transform — giống hệt ConvNeXt V2 pipeline
train_transform = A.Compose([
    RandomResizedCropRect(CONFIGURATION['image_size']),
    GaussianNoise(p=0.2),
    A.HorizontalFlip(p=0.5),
])
test_transform = A.Compose([
    A.Resize(CONFIGURATION['image_size'], CONFIGURATION['image_size']),
])

# Training dataloader — probe_split.csv dùng để monitor AUC trong vòng lặp train
train_dl, test_dl, _ = create_multitask_datafetcher(
    CONFIGURATION, train_transform, test_transform, 'train_split.csv', 'probe_split.csv'
)
print(f'Train batches: {len(train_dl)} | Test batches (probe): {len(test_dl)}')

# Gallery-probe loaders — CHỈ dùng cho đánh giá cuối, không dùng trong training
# gallery_split.csv: reference set | probe_split.csv: query set
gallery_dl, probe_dl = create_eval_loaders(CONFIGURATION, test_transform)

## 4. Teacher Model (ConvNeXt V2 — Frozen)

Load checkpoint đã train, freeze toàn bộ weights.  
Chỉ dùng `get_embedding(x)[-1]` để lấy **ID embedding** (`x_id`, tầng embedding cuối).

In [7]:
if not os.path.exists(TEACHER_CKPT):
    raise FileNotFoundError(
        f'Không tìm thấy teacher checkpoint: {TEACHER_CKPT}\n'
        f'Kiểm tra lại biến TEACHER_CKPT.'
    )

# Khởi tạo teacher với đúng kiến trúc
teacher = MTLFaceRecognition(
    backbone=CONFIGURATION['teacher_backbone'],
    num_classes=CONFIGURATION['num_classes'],
)

# Load weights
ckpt = torch.load(TEACHER_CKPT, map_location=device, weights_only=False)
state_dict = ckpt['model_state_dict']

# Remove keys related to the id_head.maglinear layer to avoid size mismatch errors
# This layer is not needed as the teacher is used for feature extraction only
keys_to_remove = [k for k in state_dict.keys() if 'id_head.maglinear' in k]
for k in keys_to_remove:
    del state_dict[k]

teacher.load_state_dict(ckpt['model_state_dict'], strict=False)  # strict=False để bỏ qua nếu có layers mới (như classifier)
print(f"Teacher loaded — epoch {ckpt.get('epoch', '?')}")

# Freeze hoàn toàn
teacher.to(device)
teacher.eval()
for p in teacher.parameters():
    p.requires_grad = False

teacher_params = sum(p.numel() for p in teacher.parameters())
print(f'Teacher params: {teacher_params:,} (tất cả frozen)')

# Smoke test — kiểm tra embedding shape
with torch.no_grad():
    _dummy = torch.randn(2, 3, 112, 112).to(device)
    # get_embedding() trả về (x_spec, x_fh, x_pose, x_emot, x_gender, x_id)
    _t_emb = teacher.get_embedding(_dummy)[-1]   # x_id
    print(f'Teacher ID embedding shape: {_t_emb.shape}')   # mong đợi [2, 512]

model.safetensors:   0%|          | 0.00/115M [00:00<?, ?B/s]

Teacher loaded — epoch 28
Teacher params: 45,670,280 (tất cả frozen)
Teacher ID embedding shape: torch.Size([2, 512])


## 4.1. Đánh giá Teacher Model (baseline)

Đo AUC của teacher trên train/test set **trước khi distillation** để có baseline so sánh với student sau này.

In [ ]:
class _TeacherEvalWrapper(torch.nn.Module):
    """MTLFaceRecognition.get_embedding() trả về tuple — wrapper giữ lại x_id (index -1)."""
    def __init__(self, teacher):
        super().__init__()
        self._teacher = teacher

    def get_embedding(self, x):
        return self._teacher.get_embedding(x)[-1]   # x_id: [B, 512]


teacher_wrapper = _TeacherEvalWrapper(teacher).to(device)
teacher_wrapper.eval()

teacher_gp_auc   = compute_id_auc_gallery_probe(gallery_dl, probe_dl, teacher_wrapper, device)
teacher_gp_rank1 = compute_rank1_gallery_probe(gallery_dl, probe_dl, teacher_wrapper, device)

rows = [
    ['Cosine AUC    (gallery→probe)', f"{teacher_gp_auc['id_cosine']:.4f}"],
    ['Euclidean AUC (gallery→probe)', f"{teacher_gp_auc['id_euclidean']:.4f}"],
    ['Rank-1 Acc    (gallery→probe)', f"{teacher_gp_rank1:.4f}"],
]
print(f"Teacher ({CONFIGURATION['teacher_backbone']}) — weight: {TEACHER_CKPT}")
print(tabulate(rows, headers=['Metric', 'Value'], tablefmt='fancy_grid'))

## 5. Student Model (MobileNetV3 — Trainable)

In [9]:
student = FaceRecognitionMobileNetV3(
    num_classes=CONFIGURATION['num_classes'],
    backbone=CONFIGURATION['backbone'],
)
student.to(device)

total_p     = sum(p.numel() for p in student.parameters())
trainable_p = sum(p.numel() for p in student.parameters() if p.requires_grad)
print(f'Student total params    : {total_p:,}')
print(f'Student trainable params: {trainable_p:,}')

# Smoke test
with torch.no_grad():
    _s_emb = student.get_embedding(_dummy)
    print(f'Student embedding shape: {_s_emb.shape}')   # mong đợi [2, 512]

model.safetensors:   0%|          | 0.00/22.1M [00:00<?, ?B/s]

Student total params    : 3,646,256
Student trainable params: 3,646,256
Student embedding shape: torch.Size([2, 512])


## 6. Knowledge Distillation Loss

```
L_total = α · L_MagFace  +  β · L_KD

L_MagFace : WeightClassMagFace — phân biệt class boundaries cho student
L_KD      : Cosine distance giữa student emb và teacher ID emb
           = mean(1 - cos_sim(norm(s_emb), norm(t_emb)))
```

**Tại sao cosine loss:**  
Face verification so sánh *hướng* của embedding (cosine sim), không phải magnitude.  
Cosine loss kéo student về đúng hướng mà không ràng buộc magnitude.

In [10]:
class KDLoss(nn.Module):
    """
    L_total = task_weight * L_MagFace  +  kd_weight * L_KD_cosine
    """

    def __init__(self, metadata_path: str, task_weight: float = 1.0, kd_weight: float = 1.0):
        super().__init__()
        self.magface = WeightClassMagLoss(metadata_path)
        self.task_w  = task_weight
        self.kd_w    = kd_weight

    def forward(
        self,
        student_logits,   # [cos_theta, cos_theta_m] từ MagLinear
        student_norm,     # x_norm từ MagLinear
        student_emb,      # [B, 512] — embedding trước MagLinear
        teacher_emb,      # [B, 512] — ID embedding của teacher (no_grad)
        id_labels,        # [B] — ground-truth identity
    ):
        # Task loss
        l_task = self.magface(student_logits, id_labels, student_norm)

        # KD loss — cosine distance trên unit hypersphere
        s_n = F.normalize(student_emb, p=2, dim=1)
        t_n = F.normalize(teacher_emb, p=2, dim=1)
        l_kd = (1.0 - F.cosine_similarity(s_n, t_n, dim=1)).mean()

        total = self.task_w * l_task + self.kd_w * l_kd
        return total, l_task, l_kd


criterion = KDLoss(
    metadata_path=train_csv,
    task_weight=CONFIGURATION['task_weight'],
    kd_weight=CONFIGURATION['kd_weight'],
)
print('KDLoss khởi tạo thành công.')

KDLoss khởi tạo thành công.


## 7. Training

In [11]:
def train_epoch(train_dl, teacher, student, criterion, optimizer, device):
    student.train()
    # teacher.eval() + frozen — không cần đặt lại mỗi epoch

    total_loss = total_task = total_kd = 0.0

    for X, y in train_dl:
        X, y = X.to(device), y.to(device)
        id_labels = y[:, 0]   # chỉ cần identity label

        # Teacher forward — no gradient, không update weights
        with torch.no_grad():
            # get_embedding() → (x_spec, x_fh, x_pose, x_emot, x_gender, x_id)
            teacher_emb = teacher.get_embedding(X)[-1]   # x_id: [B, 512]

        # Student forward — một lần duy nhất để lấy cả embedding lẫn logits
        feat         = student.backbone(X)          # [B, 512, H, W]
        student_emb  = student.embedding(feat)      # [B, 512]
        logits, norm = student.maglinear(student_emb)

        # Loss tổng hợp
        loss, l_task, l_kd = criterion(
            logits, norm, student_emb, teacher_emb, id_labels
        )

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        total_task += l_task.item()
        total_kd   += l_kd.item()

    n = len(train_dl)
    return total_loss / n, total_task / n, total_kd / n


def display_metrics(epoch, train_metrics, test_metrics):
    rows = []
    for k in train_metrics:
        tv = train_metrics[k]
        ev = test_metrics.get(k, '-')
        fmt = lambda v: f'{v:.4f}' if isinstance(v, float) else str(v)
        rows.append([k, fmt(tv), fmt(ev)])
    print(f'\nEp {epoch}:')
    print(tabulate(rows, headers=['Metric', 'Train', 'Test'], tablefmt='fancy_grid'))

In [12]:
optimizer = Adam(student.parameters(), lr=CONFIGURATION['base_lr'])
scheduler = CosineAnnealingWarmRestarts(optimizer, T_0=20, T_mult=2, eta_min=1e-6)

manager = ExperimentManager(CONFIGURATION)
manager.log_text(
    f"Teacher: {CONFIGURATION['teacher_backbone']} | "
    f"Student: {CONFIGURATION['backbone']} | "
    f"Modality: {CONFIGURATION['type']} | "
    f"kd_weight={CONFIGURATION['kd_weight']} task_weight={CONFIGURATION['task_weight']}"
)

ckpt_saver = ModelCheckpoint(
    output_dir=manager.ckpt_dir,
    mode='max',
    best_metric_name='auc_id_cosine',
)
early_stopping = MultiMetricEarlyStopping(
    monitor_keys=['auc_id_cosine'],
    patience=10,
    mode='max',
    verbose=1,
    save_dir=manager.ckpt_dir,
    start_from_epoch=5,
)

writer = SummaryWriter(log_dir=manager.log_dir)
print(f'Experiment dir: {manager.exp_dir}')
print(f'Checkpoint dir: {manager.ckpt_dir}')
print('(Tất cả được ghi thẳng vào Google Drive)')

KHOI TAO THI NGHIEM: KD_ConvNextV2_to_MobileNetV3_Albedo
Luu tru tai: /content/drive/MyDrive/experiments/KD_ConvNextV2_to_MobileNetV3_Albedo
Thoi gian: 2026-05-24 09:04:38
--------------------------------------------------
Teacher: convnextv2_tiny | Student: mobilenetv3_large_100 | Modality: albedo | kd_weight=1.0 task_weight=1.0
Experiment dir: /content/drive/MyDrive/experiments/KD_ConvNextV2_to_MobileNetV3_Albedo
Checkpoint dir: /content/drive/MyDrive/experiments/KD_ConvNextV2_to_MobileNetV3_Albedo/checkpoints
(Tất cả được ghi thẳng vào Google Drive)


In [13]:
START_EPOCH = 0

manager.log_text('BAT DAU KNOWLEDGE DISTILLATION')

for epoch in range(START_EPOCH, CONFIGURATION['epochs']):
    manager.log_text(f'\n--- Epoch {epoch+1}/{CONFIGURATION["epochs"]} ---')

    # Train
    train_loss, train_task, train_kd = train_epoch(
        train_dl, teacher, student, criterion, optimizer, device
    )

    # AUC
    train_auc = compute_id_auc(train_dl, student, device)
    test_auc  = compute_id_auc(test_dl,  student, device)

    train_metrics = {
        'loss':             train_loss,
        'loss_task':        train_task,
        'loss_kd':          train_kd,
        'auc_id_cosine':    train_auc['id_cosine'],
        'auc_id_euclidean': train_auc['id_euclidean'],
    }
    test_metrics = {
        'auc_id_cosine':    test_auc['id_cosine'],
        'auc_id_euclidean': test_auc['id_euclidean'],
    }

    # TensorBoard
    writer.add_scalar('Loss/total', train_loss, epoch + 1)
    writer.add_scalar('Loss/task',  train_task, epoch + 1)
    writer.add_scalar('Loss/kd',    train_kd,   epoch + 1)
    writer.add_scalars('AUC/cosine',
        {'train': train_auc['id_cosine'],    'test': test_auc['id_cosine']},    epoch + 1)
    writer.add_scalars('AUC/euclidean',
        {'train': train_auc['id_euclidean'], 'test': test_auc['id_euclidean']}, epoch + 1)

    # Console + file log
    display_metrics(epoch + 1, train_metrics, test_metrics)
    manager.log_metrics(epoch + 1, {**train_metrics, **test_metrics})

    # Checkpoint + Scheduler
    ckpt_saver(student, optimizer, epoch + 1, test_metrics, scheduler)
    early_stopping(test_metrics, student,epoch+1)
    scheduler.step(epoch)

    if early_stopping.early_stop:
        manager.log_text('Early stopping triggered.')
        break

writer.close()
manager.log_text('KNOWLEDGE DISTILLATION HOAN TAT.')

BAT DAU KNOWLEDGE DISTILLATION

--- Epoch 1/40 ---

Ep 1:
╒══════════════════╤═════════╤════════╕
│ Metric           │   Train │ Test   │
╞══════════════════╪═════════╪════════╡
│ loss             │ 30.0698 │ -      │
├──────────────────┼─────────┼────────┤
│ loss_task        │ 29.0684 │ -      │
├──────────────────┼─────────┼────────┤
│ loss_kd          │  1.0014 │ -      │
├──────────────────┼─────────┼────────┤
│ auc_id_cosine    │  0.9102 │ 0.8888 │
├──────────────────┼─────────┼────────┤
│ auc_id_euclidean │  0.9102 │ 0.8888 │
╘══════════════════╧═════════╧════════╛
Ep 1: loss: 30.0698, loss_task: 29.0684, loss_kd: 1.0014, auc_id_cosine: 0.8888, auc_id_euclidean: 0.8888
--> SAVE BEST MODEL (auc_id_cosine: 0.8888)

--- Epoch 2/40 ---

Ep 2:
╒══════════════════╤═════════╤════════╕
│ Metric           │   Train │ Test   │
╞══════════════════╪═════════╪════════╡
│ loss             │ 32.6391 │ -      │
├──────────────────┼─────────┼────────┤
│ loss_task        │ 31.6386 │ -      │
├────

## 8. Resume Training từ Checkpoint

> Chạy cell này khi Colab disconnect và muốn tiếp tục train.  
> Checkpoint đã được ghi thẳng vào Drive nên không bị mất.
>
> **Cách dùng:** Chạy cell Setup → Imports → Data → Teacher → Student → Loss → Setup Train,  
> sau đó chạy cell này (không chạy cell fit ở trên), rồi chạy lại cell fit.

In [ ]:
CKPT_PATH = os.path.join(manager.ckpt_dir, 'last_model.pth')

if not os.path.exists(CKPT_PATH):
    raise FileNotFoundError(f'Không tìm thấy checkpoint: {CKPT_PATH}')

checkpoint = torch.load(CKPT_PATH, map_location=device)
student.load_state_dict(checkpoint['model_state_dict'])
optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
if 'scheduler_state_dict' in checkpoint:
    scheduler.load_state_dict(checkpoint['scheduler_state_dict'])

START_EPOCH = checkpoint['epoch']
print(f'Resume từ epoch {START_EPOCH} — chạy lại cell "cell-fit" để tiếp tục.')

## 9. Đánh giá Final

In [ ]:
best_ckpt_path = os.path.join(manager.ckpt_dir, 'best_model.pth')
best_ckpt = torch.load(best_ckpt_path, map_location=device, weights_only=False)
student.load_state_dict(best_ckpt['model_state_dict'])
student.eval()
print(f"Best model từ epoch {best_ckpt.get('epoch', '?')}")

student_gp_auc   = compute_id_auc_gallery_probe(gallery_dl, probe_dl, student, device)
student_gp_rank1 = compute_rank1_gallery_probe(gallery_dl, probe_dl, student, device)

rows = [
    ['Cosine AUC    (gallery→probe)', f"{student_gp_auc['id_cosine']:.4f}"],
    ['Euclidean AUC (gallery→probe)', f"{student_gp_auc['id_euclidean']:.4f}"],
    ['Rank-1 Acc    (gallery→probe)', f"{student_gp_rank1:.4f}"],
]
print(f"\nStudent ({CONFIGURATION['backbone']}) — KD from {CONFIGURATION['teacher_backbone']}")
print(tabulate(rows, headers=['Metric', 'Value'], tablefmt='fancy_grid'))

# So sánh Teacher vs Student
compare_rows = [
    ['Model',                          CONFIGURATION['teacher_backbone'],           CONFIGURATION['backbone']],
    ['Params',                         f'{teacher_params:,}',                       f'{total_p:,}'],
    ['Cosine AUC    (gallery→probe)',   f"{teacher_gp_auc['id_cosine']:.4f}",        f"{student_gp_auc['id_cosine']:.4f}"],
    ['Euclidean AUC (gallery→probe)',   f"{teacher_gp_auc['id_euclidean']:.4f}",     f"{student_gp_auc['id_euclidean']:.4f}"],
    ['Rank-1 Acc    (gallery→probe)',   f"{teacher_gp_rank1:.4f}",                   f"{student_gp_rank1:.4f}"],
]
print('\n--- Teacher vs Student (KD) ---')
print(tabulate(compare_rows, headers=['Metric', 'Teacher', 'Student (KD)'], tablefmt='fancy_grid'))

## 10. Export ONNX

Export phần inference (backbone + embedding + L2 normalize, bỏ MagLinear)  
để chuẩn bị quantize INT8 deploy edge device.

In [ ]:
class InferenceWrapper(nn.Module):
    """Backbone + embedding + L2 normalize — không có MagLinear."""
    def __init__(self, model):
        super().__init__()
        self.backbone  = model.backbone
        self.embedding = model.embedding

    def forward(self, x):
        emb = self.embedding(self.backbone(x))
        return F.normalize(emb, p=2, dim=1)


inference_model = InferenceWrapper(student).eval().cpu()
dummy_input = torch.randn(1, 3, 112, 112)

# Lưu vào Drive để không mất
onnx_path = os.path.join(manager.ckpt_dir, 'kd_mobilenetv3_fr.onnx')

torch.onnx.export(
    inference_model,
    dummy_input,
    onnx_path,
    input_names=['input'],
    output_names=['embedding'],
    dynamic_axes={'input': {0: 'batch'}, 'embedding': {0: 'batch'}},
    opset_version=17,
)
print(f'Exported ONNX: {onnx_path}')